# 11 — Validate susceptibility mart

## Goal

`mart_susceptibility` is the patient-deduplicated table I plan to use for Tableau. Its grain is:

`source release + cohort + reporting organism + reporting phenotype + antibiotic`

CRE and MDR *P. aeruginosa* are intentionally excluded from this mart and handled in `mart_mdro`.

In [ ]:
%sql
select
    source_version,
    cohort_id,
    organism_reporting_group,
    organism_name,
    reporting_phenotype,
    antibiotic,
    count(*) as row_count

from workspace.micro_dev.mart_susceptibility

group by
    source_version,
    cohort_id,
    organism_reporting_group,
    organism_name,
    reporting_phenotype,
    antibiotic

having count(*) > 1;

### Unique mart grain

This should return no rows. Every organism/phenotype/antibiotic combination should appear once per cohort and source release.

In [ ]:
%sql
select
    cohort_id,
    organism_name,
    reporting_phenotype,
    antibiotic,
    eligible_count,
    tested_endpoint_count,
    untested_count

from workspace.micro_dev.mart_susceptibility

where tested_endpoint_count + untested_count <> eligible_count;

### Eligible = tested + untested

This checks the first denominator split. Every eligible reporting isolate should either have an endpoint for the drug or be counted as untested.

In [ ]:
%sql
select
    cohort_id,
    organism_name,
    reporting_phenotype,
    antibiotic,
    tested_endpoint_count,
    categorized_count,
    susceptible_count,
    resistant_count,
    intermediate_count,
    inconclusive_count,
    synergism_count,
    missing_result_count,
    conflict_result_count

from workspace.micro_dev.mart_susceptibility

where tested_endpoint_count <>
      categorized_count
    + inconclusive_count
    + synergism_count
    + missing_result_count
    + conflict_result_count

   or categorized_count <>
      susceptible_count
    + resistant_count
    + intermediate_count;

### Tested-result reconciliation

Tested endpoints have to reconcile to categorized S/R/I, inconclusive, synergism, missing, or conflict. The categorized subset also has to equal S + R + I.

In [ ]:
%sql
select
    cohort_id,
    organism_name,
    reporting_phenotype,
    antibiotic,
    eligible_count,
    categorized_count,
    susceptible_count,
    resistant_count,
    intermediate_count

from workspace.micro_dev.mart_susceptibility

where categorized_count > eligible_count
   or susceptible_count > categorized_count
   or resistant_count > categorized_count
   or intermediate_count > categorized_count;

### Basic count bounds

No analytical count should be larger than its parent denominator.

In [ ]:
%sql
with expected_pair_counts as (

    select
        organism_reporting_group,
        count(*) as expected_antibiotic_count

    from workspace.micro_dev.reportable_organism_antibiotic_pairs

    where include_primary_dashboard = 1
      and review_status = 'reviewed'

    group by organism_reporting_group

),

actual_pair_counts as (

    select
        cohort_id,
        organism_reporting_group,
        organism_name,
        reporting_phenotype,
        count(distinct antibiotic) as actual_antibiotic_count

    from workspace.micro_dev.mart_susceptibility

    group by
        cohort_id,
        organism_reporting_group,
        organism_name,
        reporting_phenotype

)

select
    a.cohort_id,
    a.organism_reporting_group,
    a.organism_name,
    a.reporting_phenotype,
    e.expected_antibiotic_count,
    a.actual_antibiotic_count

from actual_pair_counts a

inner join expected_pair_counts e
    on a.organism_reporting_group = e.organism_reporting_group

where a.actual_antibiotic_count <> e.expected_antibiotic_count;

### Reporting-pair coverage

Each reporting phenotype inherits the reviewed antibiotic set for its base `organism_reporting_group`. This check makes sure the mart has the full reviewed pair set wherever that organism is represented.

In [5]:
%sql
select
    count(*) as mart_rows,

    count(
        distinct susceptibility_mart_key
    ) as unique_keys,

    sum(
        case
            when eligible_count
                 <> categorized_count
                  + inconclusive_count
                  + synergism_count
                  + missing_result_count
                  + conflict_result_count
                  + untested_count
                then 1
            else 0
        end
    ) as denominator_mismatches,

    sum(
        case
            when reporting_phenotype in (
                'cre',
                'mdr_p_aeruginosa'
            )
                then 1
            else 0
        end
    ) as excluded_mdro_rows

from workspace.micro_dev.mart_susceptibility

,mart_rows,unique_keys,denominator_mismatches,excluded_mdro_rows
0,337,337,0,0


### Structural mart check

The mart has 337 rows and 337 unique keys. There are no denominator mismatches and no CRE/MDR rows leaking into the main susceptibility output.

In [1]:
%sql
with expected as (

    select
        source_version,
        cohort_id,
        organism_reporting_group,
        organism_name,
        reporting_phenotype,

        count(*) as expected_eligible_count

    from workspace.micro_dev.int_first_reporting_isolate

    where include_main_antibiogram = true

    group by
        source_version,
        cohort_id,
        organism_reporting_group,
        organism_name,
        reporting_phenotype

),

actual as (

    select
        source_version,
        cohort_id,
        organism_reporting_group,
        organism_name,
        reporting_phenotype,

        min(eligible_count) as min_mart_eligible_count,
        max(eligible_count) as max_mart_eligible_count,

        count(*) as antibiotic_rows

    from workspace.micro_dev.mart_susceptibility

    group by
        source_version,
        cohort_id,
        organism_reporting_group,
        organism_name,
        reporting_phenotype

),

comparison as (

    select
        e.*,

        a.min_mart_eligible_count,
        a.max_mart_eligible_count,
        a.antibiotic_rows,

        case
            when a.organism_reporting_group is null
                then 'no_reviewed_antibiotic_pairs'

            when e.expected_eligible_count
                 <> a.min_mart_eligible_count
                then 'minimum_denominator_mismatch'

            when e.expected_eligible_count
                 <> a.max_mart_eligible_count
                then 'maximum_denominator_mismatch'

            else 'pass'
        end as validation_status

    from expected e

    left join actual a
        on e.source_version = a.source_version
       and e.cohort_id = a.cohort_id
       and e.organism_reporting_group =
           a.organism_reporting_group
       and e.organism_name =
           a.organism_name
       and e.reporting_phenotype =
           a.reporting_phenotype

)

select
    validation_status,
    count(*) as reporting_categories,

    sum(expected_eligible_count)
        as first_reporting_isolates

from comparison

group by validation_status

order by validation_status

,validation_status,reporting_categories,first_reporting_isolates
0,no_reviewed_antibiotic_pairs,13,3354
1,pass,56,73491


### Denominator back-check

I rebuild the expected eligible count from `int_first_reporting_isolate` and compare it with every antibiotic row in the mart.

Fifty-six reporting categories match exactly. Thirteen categories have no reviewed antibiotic pairs, which is intentional; I leave them out rather than invent a panel.

In [ ]:
%sql
select
    cohort_id,
    organism_name,
    reporting_phenotype,
    antibiotic,
    categorized_count,
    display_status

from workspace.micro_dev.mart_susceptibility

where (
        categorized_count >= 30
        and display_status <> 'display'
      )
   or (
        categorized_count < 30
        and display_status <> 'insufficient_data'
      );

### Display threshold

The project uses a simple reporting threshold of 30 categorized S/R/I results. This is a project display rule, not a claim of formal guideline compliance. The query should return no rows.

In [ ]:
%sql
select
    organism_reporting_group,
    organism_name,
    reporting_phenotype,
    antibiotic,
    eligible_count,
    tested_endpoint_count,
    untested_count,
    categorized_count,
    susceptible_count,
    resistant_count,
    intermediate_count,
    inconclusive_count,
    synergism_count,
    missing_result_count,
    conflict_result_count,
    categorized_result_coverage_pct,
    susceptible_pct,
    resistant_pct,
    display_status

from workspace.micro_dev.mart_susceptibility

where cohort_id = 'blood'

order by
    organism_reporting_group,
    reporting_phenotype,
    pair_order;

### Inspect the blood output

This is the table shape that will feed the first Tableau view. I keep denominator, testing coverage, S/R/I counts, exceptions, and display status together so the visualization does not have to reconstruct the logic.

In [ ]:
%sql
select *

from workspace.micro_dev.mart_susceptibility

where cohort_id = 'blood'
  and organism_reporting_group = 'Escherichia coli'
  and antibiotic = 'Ceftriaxone'

order by reporting_phenotype;

### Trace one endpoint

I use blood *E. coli* / ceftriaxone as a spot check because it can appear in both the base and ESCE-R reporting categories. Looking at both rows is a quick way to confirm that phenotype-aware deduplication is carrying through to the mart.

### Takeaway

The main susceptibility mart reconciles back to the reporting-isolate layer and keeps its denominator logic explicit. It is ready to use as the Tableau data source.